<div style="background-color:#000;"><img src="pqn.png"></img></div><div><a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.</div>

## Library installation

This installs the only external library needed to run the notebook.

In [ ]:
!pip install numpy

## Imports and setup

NumPy provides the math functions and array operations we use to build the binomial price tree and compute option values efficiently.

In [1]:
import numpy as np

## Define the American option pricer

This function implements the Cox-Ross-Rubinstein binomial tree model, which is the standard method for pricing American-style options because it can evaluate early exercise at every node.

In [2]:
def american_option_pricer(
    spot, strike, rate, vol, expiry, steps, option_type
):
    dt = expiry / steps
    u = np.exp(vol * np.sqrt(dt))
    d = 1 / u

    p = (np.exp(rate * dt) - d) / (u - d)

    price_tree = np.zeros((steps + 1, steps + 1))
    for i in range(steps + 1):
        price_tree[i, -1] = spot * (u ** (steps - i)) * (d**i)

    option_tree = np.zeros_like(price_tree)
    if option_type.lower() == "call":
        option_tree[:, -1] = np.maximum(
            price_tree[:, -1] - strike, 0
        )
    elif option_type.lower() == "put":
        option_tree[:, -1] = np.maximum(
            strike - price_tree[:, -1], 0
        )
    else:
        raise ValueError(
            "Option type must be either 'call' or 'put'."
        )

    for t in range(steps - 1, -1, -1):
        for i in range(t + 1):
            exercise = 0
            if option_type.lower() == "call":
                exercise = price_tree[i, t] - strike
            elif option_type.lower() == "put":
                exercise = strike - price_tree[i, t]

            hold = np.exp(-rate * dt) * (
                p * option_tree[i, t + 1]
                + (1 - p) * option_tree[i + 1, t + 1]
            )
            option_tree[i, t] = np.maximum(exercise, hold)

    return option_tree[0, 0]

The function first splits the time to expiry into discrete steps and computes the up and down factors that determine how the stock price can move at each step. It then fills in the terminal payoffs at expiration and walks backward through the tree, comparing the value of exercising immediately against the discounted expected value of holding. That comparison at every single node is what makes this model correct for American options and what Black-Scholes cannot do.

## Price an American call option

We call the pricer with concrete market parameters to get the fair value of a 1-year American call on a $55 stock with a $50 strike, 5% risk-free rate, and 30% volatility, using 100 time steps for accuracy.

In [3]:
option_price = american_option_pricer(
    spot=55.0,
    strike=50.0,
    rate=0.05,
    vol=0.3,
    expiry=1.0,
    steps=100,
    option_type="call",
)

Increasing the steps parameter gives a more accurate price because the tree approximates continuous stock movement more closely, but it also increases computation time. With 100 steps, we get a good balance between precision and speed. Try changing option_type to "put" or adjusting vol and expiry to see how the fair value shifts, which builds intuition for how these inputs drive option prices in real markets.

<a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.